# 📺 Ingest a YouTube Live Stream

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/real_time_streaming/YouTube_Live_Ingestion.ipynb)

Connect any YouTube **live** broadcast to VideoDB as an RTStream, watch clips of it while it runs, then export the recording as a regular VideoDB video.

You will:
1. Connect a YouTube live URL as an RTStream.
2. Check the stream is receiving media.
3. Watch the last two minutes as a playable clip.
4. Stop the stream and export the recording.

**Requirements**
- A VideoDB **Pro plan**. YouTube live sources are billed **$2.084 per hour**, stream compute included. On the Free plan, connecting a YouTube URL returns a `pro_plan_required` error.
- `videodb` SDK **0.5.4** or newer.

**Good to know**
- Only live broadcasts work, not recorded videos. Up to 720p is ingested.
- With `store=True`, keep one session to **4 hours** at most. For a longer event, export, then connect again.
- For a live transcript with speaker labels, see [Live Transcription with Speaker Labels](./Live_Transcription_Speaker_Labels.ipynb).

---
## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q "videodb>=0.5.4"

---
## 🔑 Step 2: Connect to VideoDB

Enter your `VIDEO_DB_API_KEY` when asked. The input is masked.

In [ ]:
import os
from getpass import getpass

import videodb

if not os.environ.get("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Please enter your VideoDB API Key: ")

conn = videodb.connect()
coll = conn.get_collection()
print("Connected to VideoDB")

---
## 📡 Step 3: Connect the YouTube Live Stream

Use any YouTube live link: a `watch?v=` link, a `/live/` link, or a channel's `/@name/live` page. We use DW News.

`store=True` keeps a recording you can export later.

In [ ]:
YOUTUBE_LIVE_URL = "https://www.youtube.com/@dwnews/live"
STREAM_NAME = "DW News live"

try:
    stream = coll.connect_rtstream(
        name=STREAM_NAME,
        url=YOUTUBE_LIVE_URL,
        media_types=["video", "audio"],
        store=True,
    )
except Exception as error:
    # On the Free plan: "YouTube links require the Pro plan. Upgrade to Pro to continue."
    raise SystemExit(f"Could not connect the stream: {error}")

print(stream.id, stream.status)

#### Already connected the stream? Reconnect with its id instead.

In [ ]:
# stream = coll.get_rtstream("rts-...")

---
## ⏳ Step 4: Let Media Arrive

The stream reports `connected` right away. Its worker then resolves the YouTube page and starts receiving media, usually within one to two minutes. Give it about three minutes before asking for a clip.

In [ ]:
import time

WAIT_MINUTES = 3
for remaining in range(WAIT_MINUTES * 60, 0, -30):
    print(f"Receiving media... {remaining} s to go")
    time.sleep(30)
print("Ready")

---
## ▶️ Step 5: Watch the Last Two Minutes

`generate_stream(start, end)` turns any time range of the live stream into a playable clip. Times are Unix seconds.

In [ ]:
from IPython.display import IFrame, display

end = int(time.time())
start = end - 120
player_url = stream.generate_stream(start=start, end=end)
print(player_url)
display(IFrame(player_url, width=720, height=405))

---
## ⏹️ Step 6: Stop the Stream

Stopping ends the recording and the billing. Always stop a stream you no longer need.

In [ ]:
stream.stop()
print("Stream stopped")

---
## 🎬 Step 7: Export the Recording

Export turns the stopped stream's recording into a regular VideoDB video. It runs in the background: call `export()` again until it returns a `video_id`.

In [ ]:
import time

result = None
for _ in range(60):
    try:
        result = stream.export(name=STREAM_NAME)
        if getattr(result, "video_id", None):
            break
    except Exception as error:
        print("Waiting for the recording:", error)
    time.sleep(10)

if not getattr(result, "video_id", None):
    raise SystemExit("The recording is not ready yet: run this cell again in a minute.")
video = coll.get_video(result.video_id)
print("Exported:", video.id)
print(video.player_url)

---
## ✅ What You Built

- A YouTube live broadcast ingested as an RTStream.
- A clip of any moment while it was live.
- The full recording in your collection, ready to index, search and clip like any other video.

Next: add a live transcript with speakers in [Live Transcription with Speaker Labels](./Live_Transcription_Speaker_Labels.ipynb), or attach live alerts with the other notebooks in `real_time_streaming/`.